# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library, specifically using the FAIR^2 mass spectrometry dataset of extracellular vesicles.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata_json = json.loads(dataset.metadata.to_jsonld())

dataset_name = dataset.metadata.name
dataset_description = dataset.metadata.description

print(f"Dataset: {dataset_name}\nDescription: {dataset_description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

In [ ]:
# List all RecordSets with their @id and fields
record_sets = list(dataset.metadata.record_sets)

print(f"Number of record sets found: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet Name: {rs.name}")
    print(f"\t@id: {rs.id}")
    print(f"\tDescription: {rs.description if hasattr(rs, 'description') else 'N/A'}")
    print("\tFields:")
    for field in rs.fields:
        print(f"\t  - {field.name}: @id={field.id}, dataType={field.data_type}")
    print('-' * 60)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All record sets and field `@id`s are referenced by their unique `@id` as required.

We will extract all available record sets.

In [ ]:
# Extract data from each record set by @id
dataframes = {}
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet @id={record_set_id} ...")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        print(f"Loaded {len(df)} records. Columns: {df.columns.tolist()}")
        dataframes[record_set_id] = df
    except Exception as e:
        print(f"Failed to load records for {record_set_id}: {e}")
    print('-' * 60)

# Show the columns of the main record set (assume the first for this example)
main_record_set_id = record_set_ids[0]
print(f"Main RecordSet @id: {main_record_set_id}")
print(f"Fields: {dataframes[main_record_set_id].columns.tolist()}")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data.

For illustration, let's pick a common numeric field such as coverage percentage or molecular weight (MW) and a grouping field (e.g., peptide count or accession field) by their `@id`.

In [ ]:
# Select a numeric field and a group field by their field @id
df = dataframes[main_record_set_id]
# We'll inspect available fields
print("Fields in dataframe:", df.columns.tolist())

# Try to automatically select appropriate fields
numeric_field_candidates = [c for c in df.columns if 'mw' in c.lower() or 'coverage' in c.lower() or 'peptide' in c.lower() or 'abundance' in c.lower()]
group_field_candidates = [c for c in df.columns if 'accession' in c.lower() or 'group' in c.lower() or 'description' in c.lower()]

if numeric_field_candidates:
    numeric_field = numeric_field_candidates[0]
else:
    numeric_field = df.select_dtypes(include=['number']).columns[0]

if group_field_candidates:
    group_field = group_field_candidates[0]
else:
    group_field = None

print(f"Numeric field selected: {numeric_field}")
if group_field:
    print(f"Group field selected: {group_field}")

# Remove outliers and sanitize data
df_num = pd.to_numeric(df[numeric_field], errors='coerce')
threshold = df_num.mean() + 2 * df_num.std()
filtered_df = df[df_num > threshold]

print(f"Filtered records with {numeric_field} > {threshold:.2f}: {len(filtered_df)} records")
print(filtered_df[[numeric_field]].head())

# Normalize selected numeric field
filtered_df = filtered_df.copy()
filtered_df[f"{numeric_field}_normalized"] = (pd.to_numeric(filtered_df[numeric_field], errors='coerce') - df_num.mean()) / df_num.std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by the group_field and get the mean, if available
if group_field and group_field in df.columns:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"\nGrouped data by {group_field} (showing mean of numeric fields):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of the normalized numeric field
if not filtered_df.empty:
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[f"{numeric_field}_normalized"], bins=20, kde=True)
    plt.title(f"Distribution of Normalized {numeric_field}")
    plt.xlabel(f"{numeric_field} (normalized)")
    plt.ylabel("Count")
    plt.show()

# Scatter plot of numeric field vs group if possible
if group_field and group_field in filtered_df.columns:
    plt.figure(figsize=(12, 4))
    sns.scatterplot(data=filtered_df, x=group_field, y=numeric_field)
    plt.title(f"{numeric_field} grouped by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook demonstrated how to load, explore, and analyze records and fields from a FAIR^2 mass spectrometry dataset using the `mlcroissant` library, referencing all entities by their Croissant `@id` fields for reproducibility. Further domain analysis can be tailored to biological or biochemical questions based on the extracted and processed data.